# PANOPLY Workbench Startup Notebook

This notebook walks you through uploading, validating, and configuring PANOPLY proteogenomic
data on a **Manifold** workbench, and produces two files:

* **`master-parameters.yaml`** -- default pipeline parameters, merged with the groups, colors,
  and toggles you choose below. Same shape as the parameters file produced by the Terra-based
  [`PANOPLY-startup-notebook.ipynb`](./PANOPLY-startup-notebook.ipynb).
* **`inputs.json`** -- a Cromwell-style input file (S3 file-paths + key parameters) for a PANOPLY
  WDL workflow -- `panoply_unified_workflow` by default, or any workflow in the
  [PANOPLY GitHub repo](https://github.com/broadinstitute/PANOPLY).

-----
### Using this notebook
1. This notebook requires an **R kernel** (IRkernel). If your environment only offers a Python
   kernel, install one first from a terminal, e.g. `mamba install -c conda-forge r-irkernel`,
   then select the R kernel for this notebook.
2. Run the **Setup** cell once per session -- it installs any R packages this notebook needs
   that aren't already present. A couple are large Bioconductor packages used for gene/protein-ID
   conversion, so the very first run can take a while; subsequent runs are fast.
3. You can re-run this notebook at any time. Prior choices are saved to
   `~/workbench/.panoply-session.yaml` and reloaded automatically -- so you can pick up where you
   left off, including generating `inputs.json` for a brand-new sample subset without redoing
   anything else.
4. Run cells top to bottom the first time. Several cells prompt for input in the console below
   the cell -- read the preceding text before running each one.

### Prepare your data
Place (or upload) the following into `~/workbench/inputs/`:
* At least **one proteomics dataset** (global proteome, phosphoproteome, acetylome, and/or
  ubiquitylome), in [GCT v1.3 format](https://clue.io/connectopedia/gct_format).
* Genomics data -- CNA and/or RNA, also GCT v1.3.
* An `Annotation` CSV with at least `Sample.ID` and `Type` columns, plus any other sample
  annotations you have.
* Optionally: a `groups` CSV (one annotation-column name per line), your own parameter YAML
  (otherwise PANOPLY's default `master-parameters.yaml` is used), and/or PTM-SEA / GSEA `gmt`
  pathway databases (otherwise bundled defaults are used).

Sample IDs (GCT column names) must match the `Sample.ID`s in your annotation table. See the
[PANOPLY wiki](https://github.com/broadinstitute/PANOPLY/wiki) for more on data formats.


### Setup
Run once per session. `wb_setup()` installs any R packages this notebook needs that aren't
already present (a couple are large Bioconductor packages used for gene/protein-ID conversion,
so the very first run can take a while; subsequent runs are fast), then loads the helper module.

In [ ]:
source("workbench-src/config.r")
wb_setup()
state <- wb_load_state()

### Configuration
Two things you may want to change:
* `GITHUB_REF` -- the branch/tag of [broadinstitute/PANOPLY](https://github.com/broadinstitute/PANOPLY)
  that workflow WDLs and the default `master-parameters.yaml` are fetched from. Defaults to the
  `issue-githubWDL` branch for now; bump this once a release branch is the intended target.
* `TARGET_WORKFLOW` -- which PANOPLY workflow to build `inputs.json` for. Defaults to
  `panoply_unified_workflow`. Run `wb_list_github_workflows()` (see the *Generate inputs.json*
  section below) to see other options.

In [ ]:
GITHUB_REF      <- "issue-githubWDL"
TARGET_WORKFLOW <- "panoply_unified_workflow"

state$github_ref      <- GITHUB_REF
state$target_workflow <- TARGET_WORKFLOW
state <- wb_save_state(state)

# Inputs
Run the cell below to see the available data categories, then run `wb_load_and_map_inputs()`
to map each file you've placed in `~/workbench/inputs/` (or in a ZIP you point it at) to one of
these categories. Each category can hold only one file; mapping a second file to the same
category overwrites the first.

In [ ]:
wb_list_data_categories()

In [ ]:
# If you have a single ZIP file instead of individual files, uncomment and set zip_path:
# state <- wb_load_and_map_inputs(state, zip_path = "~/workbench/my_data.zip")
state <- wb_load_and_map_inputs(state)

# Validation
Checks the annotation table for required columns and unique `Sample.ID`s, checks sample-ID
overlap between the annotation table and every mapped GCT file, and checks (or interactively
helps you fix) the gene-ID column in each proteomics/genomics GCT file.

In [ ]:
state <- wb_validate_inputs(state)

# Data Processing
Toggles proteomics normalization/filtering, and (if the relevant data was mapped above) PTM-SEA
and MetaboAnalyst. If you opt into PTM-SEA or MetaboAnalyst, you'll be prompted to confirm or
select the relevant ID column.

In [ ]:
state <- wb_select_preprocessing_options(state)

# COSMO Label Selection (optional)
COSMO (COrrection of Sample Mislabeling by Omics) needs 1-3 clinical attributes that are binary,
well-balanced, and free of NAs. You'll be shown the valid candidates from your annotation table
and asked to pick from among them.

In [ ]:
state <- wb_select_cosmo_attributes(state)

# Clumps-PTM Setup (optional)
Only offered if at least 2 of {phosphoproteome, acetylome, ubiquitylome} were mapped above.
Requires a local reference FASTA (matching the accession-ID type used in your PTM data) --
place one under `~/workbench/inputs/` and point `fasta_path` at it below.

In [ ]:
state <- wb_select_clumpsptm_groups(state, fasta_path = "~/workbench/inputs/reference.fasta")

# Groups
Groups are the categorical annotations used for association and enrichment analysis. By default,
all valid annotation columns are used (or the columns listed in your `groups` file, if you
provided one) -- pass an explicit `columns = c(...)` to override. Annotations with more than
`max_categories` unique values are excluded (or treated as continuous, if numeric).

In [ ]:
wb_list_annotation_columns(state)

In [ ]:
# state <- wb_select_groups(state, columns = c("Type", "Stage"), max_categories = 10)
state <- wb_select_groups(state, max_categories = 10)

## Color Schemes (optional)
### See the current color scheme

In [ ]:
wb_show_colors(state)

### Reset colors to defaults
Colors are assigned automatically based on the number of unique values per group; NA is always grey.

In [ ]:
state <- wb_reset_colors(state)

### Edit a color
Look up the group and value names from `wb_show_colors(state)` above, then set a new hex color.

In [ ]:
state <- wb_edit_color(state, group = "Type", value = "Tumor", hex_color = "#E41A1C")

# Sample Subsets
Replaces Terra sample sets: each subset is a local folder under `~/workbench/subsets/<name>/`
containing the GCT/CSV files filtered down to the matching samples. An `all` subset (every
sample) is the natural starting point -- run the cell below once for it, then run the following
cell as many times as you like (with a new name/column/values) to add more subsets.

In [ ]:
state <- wb_create_subset(state, "all")

In [ ]:
state <- wb_create_subset(state, "tumor_only", filter_col = "Type", filter_vals = "Tumor")
# Re-run this cell (with a new name, filter_col, and filter_vals) for each additional subset.

# Finalize Parameters
Builds `master-parameters.yaml`: PANOPLY's default module parameters (fetched from
`GITHUB_REF`, or your own uploaded parameter file, if you provided one), merged with the
groups/colors/toggles chosen above.

In [ ]:
master_params_path <- wb_build_master_parameters_yaml(state)
master_params_path

# Workflow Run Options
These three are required, top-level toggles for `panoply_unified_workflow` that (unlike the
toggles above) aren't inferred from your data -- set them explicitly.

In [ ]:
state$toggles$run_cmap   <- FALSE  # run CMAP drug-connectivity analysis?
state$toggles$run_mo_nmf <- FALSE  # run multi-omic NMF clustering?
state$toggles$run_so_nmf <- TRUE   # run single-omic NMF clustering?
state <- wb_save_state(state)

# Generate `inputs.json`
Builds a Cromwell-style `inputs.json` for `TARGET_WORKFLOW`, with every -omics/groups/database
file path translated to its `s3://` location and the parameters set above filled in. Uninvolved
inputs (e.g. per-task memory/disk overrides) are intentionally left for `master-parameters.yaml`
or Cromwell defaults to handle.

To target a different workflow, see what's available and update `TARGET_WORKFLOW` in the
*Configuration* section above:
```r
# wb_list_github_workflows()
```

In [ ]:
inputs_path <- wb_update_inputs_json_for_subset(state, subset_name = "all")
inputs_path

# Regenerate for a different subset (or refresh a hand-edited file)
Re-running this for a new `subset_name` updates **only the file-path inputs** in `inputs.json` --
any parameter you've since hand-edited directly in the file (or added, e.g. a task-level resource
override) is left untouched. Point `existing_inputs_path` at any copy you've been editing,
including one you moved or renamed; a `.bak` backup of it is written before every update.

In [ ]:
inputs_path <- wb_update_inputs_json_for_subset(state, subset_name = "tumor_only",
                                                existing_inputs_path = inputs_path)

# Done

In [ ]:
cat("master-parameters.yaml\n  local:", master_params_path, "\n  s3:   ", wb_local_to_s3(master_params_path), "\n\n")
cat("inputs.json\n  local:", inputs_path, "\n  s3:   ", wb_local_to_s3(inputs_path), "\n")